In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from torch.utils.data import Dataset
import random
import os
import json
from tqdm import tqdm

In [9]:
def set_seed(seed=42):
    random.seed(seed)
#     np.random.seed(seed)
#     tf.random.set_seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

set_seed(42)


In [10]:

'''
load model, tokenizer
'''

def load_model():
    model_name = "Qwen/Qwen2.5-0.5B"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('using device:', device)
    model.enable_input_require_grads()

    model.to(device)
    return model, tokenizer


In [11]:
'''
data part
'''
data_dict = {"hotpot_train": "./dataset/hotpotQA/hotpot_train_v1.1.json",
               "hotpot_test": "./dataset/hotpotQA/hotpot_dev_distractor_v1.json",
               "squad_train": "./dataset/squad2.0/train-v2.0.json",
               "squad_test": "./dataset/squad2.0/dev-v2.0.json"
               }

MAX_SEQ_LEN = 3000
MAX_QES_LEN = 100
MAX_TRAIN_NUM = 10000
"""
数据格式
{"title":"", context":"", "question":"", "answer":"", answer_idx:[(start, end), (), ()]}
"""
import copy

def data_augmentation(data, corpus):
    print("doing augmentation now")
    
    cnt = 0
    new_data = []
    for example in tqdm(data):
        new_data.append(copy.deepcopy(example))
        context = copy.deepcopy(example["context"])
#         if random.random() < 0.3:
#             new_context = ""
#             for idx_pair in example["answer_idx"]:

#                 new_context += context[idx_pair[0]:idx_pair[1]]
#         else:
        new_context = ""
        supporting = []
        for idx_pair in example["answer_idx"]:
            replace_content = random.sample(corpus, 1)[0]
#                 print(type(replace_content))
            new_context += replace_content + context[idx_pair[0]:idx_pair[1]]
            supporting.append(context[idx_pair[0]:idx_pair[1]])
        
        if len(new_context) > MAX_SEQ_LEN - MAX_QES_LEN:
            continue
            
#         example["old_context"] = example["context"]
        example["context"] = new_context
        if cnt == 0:
            
            print("**************sample example**************")
            print("question:", example["question"])
            print("answer:", example["answer"])
#             print("supporting_fact:", supporting)
#             print("context:", context)
            print("before:", new_data[-1]["context"])
            print("after:", example["context"])
            cnt +=1
        new_data.append(example)
    return new_data

In [12]:
                
def hotpotQA_dataload(file_path, da = False):
    data_list = []
    max_len = 0
    corpus = []
    
    with open(file_path) as f:
        text = json.loads(f.read())
        for idx, item in tqdm(enumerate(text)):

            ans = item["answer"]
            question = item["question"]
            title = [t[0] for t in item["context"]]
            context = ["".join(t[1]) for t in item["context"]]
            corpus.extend(context)
            title2context = {}
            for _t, _c in zip(title, context):
                title2context[_t] = _c
            context = "".join(context)
            if len(context) > MAX_SEQ_LEN - MAX_QES_LEN or len(question) > MAX_QES_LEN:
                continue
            supporting_sentence = ["".join(title2context[t[0]]) for t in item["supporting_facts"]]

            answer_idx = []
            for sent in supporting_sentence:
                answer_start = context.find(sent)
                answer_idx.append((answer_start, answer_start + len(sent)))
            new_instance = {"title": ".".join(title), "context": context, "question": question, "answer": ans,
                            "answer_idx": answer_idx}
            #             if idx % 100 == 0:
            #                 print(new_instance)
            data_list.append(new_instance)
#             max_len = max(max_len, len(context + question))
    if da:
        data = data_augmentation(data_list, corpus)
        
    train_num = min(MAX_TRAIN_NUM*2 if da else MAX_TRAIN_NUM, len(data_list))
    data_list = random.sample(data_list, k=train_num)
    return data_list, corpus


def squad_dataload(file_path, da=False):
    data_list = []
    max_len = 0
    corpus = []
    
    with open(file_path) as f:
        text = json.loads(f.read())
        data = text["data"]
        for example in tqdm(data):
            title = example["title"]
            for each in example["paragraphs"]:
                context = each["context"]
                if len(context) > MAX_SEQ_LEN - MAX_QES_LEN:
                    continue
                #             print(context)
                for _v in each["qas"]:
                    try:
                        question = _v["question"]
                        if len(question) > MAX_QES_LEN:
                            continue
                        #                     print(_v["answers"])
                        answer = _v["answers"][0]["text"]
                        answer_start = _v["answers"][0]["answer_start"]
                        while answer_start >=0 and context[answer_start] not in [".", "?", "!"]:
                            answer_start -= 1
                        answer_start +=1
                        new_instance = {"title": title, "context": context, "question": question, "answer": answer,
                                        "answer_idx": [(answer_start, len(context))]}
                        #                 print(new_instance)
                        data_list.append(new_instance)
                        max_len = max(max_len, len(context + question))
                        corpus.append(context[:answer_start])

                    except Exception as e:
                        pass
    if da:
        data = data_augmentation(data_list, corpus)
        
    train_num = min(MAX_TRAIN_NUM*2 if da else MAX_TRAIN_NUM, len(data_list))
    data_list = random.sample(data_list, k=train_num)
    return data_list, corpus

prompt1 = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant. Now please answer the question according to the given context. Just give me answer without explanation. Context and question are as follows: \ncontext:"
prompt2 = "\nquestion:"
prompt3 = "\nanswer:"      
    
class CustomDataset(Dataset):
    def __init__(self, name, tokenizer, post="train", da = False):
        self.data, _ = self.dataloader(data_dict[f"{name}_{post}"], da)
        self.tokenizer = tokenizer
        self.input_data = []

        self.end_token_ids = self.tokenizer.convert_tokens_to_ids('<|endoftext|>')
        self.prompt1_tokens = self.tokenizer(prompt1, add_special_tokens=False)

        self.prompt2_tokens = self.tokenizer(prompt2, add_special_tokens=False)
        
        self.prompt3_tokens = self.tokenizer(prompt3, add_special_tokens=False)

        #         print(self.tokenizer.tokenize(prompt2))
        #         print(self.prompt2_tokens)

        self.tokenize()
    

    def dataloader(self, file_path, da = False):
        write_path = file_path+".train_"+str(int(da))
        data, corpus = [], []
        if os.path.exists(write_path):
            print("loading from:{}".format(write_path))
            with open(write_path) as f:
                for line in tqdm(f):
                    data.append(json.loads(line.strip()))
                
        else:
            print("loading from:{}".format(file_path))

            if "squad" in file_path:
                data, corpus = squad_dataload(file_path, da)

            elif "hotpot" in file_path:
                data, corpus = hotpotQA_dataload(file_path, da)
            else:
                print("invalid data data, please select from (hotpotQA, squad2.0)")
                return
                
            with open(write_path, "w") as f:
                for item in data:
                    f.write(json.dumps(item, ensure_ascii=False)+"\n")
                print("data feature write in :", write_path)

        #     print(data[:10])
        if "train" in file_path:
            print("train_num:{}".format(len(data)))
        else:
            print("eval_num:{}".format(len(data)))
        return data, corpus
    
    def tokenize(self):
        print("begin tokenize")
        for idx, item in enumerate(tqdm(self.data)):
            context_tokens = self.tokenizer(item["context"], add_special_tokens=False)
            question_tokens = self.tokenizer(item["question"], add_special_tokens=False)
            label_tokens = self.tokenizer(item["answer"], add_special_tokens=False)

            max_context_len = MAX_SEQ_LEN - len(self.prompt1_tokens["input_ids"]) - len(self.prompt2_tokens)
            max_question_len = MAX_QES_LEN - 1 - len(self.prompt3_tokens["input_ids"])

            input_ids = (
                    self.prompt1_tokens["input_ids"] +
                    context_tokens["input_ids"][:max_context_len] +
                    self.prompt2_tokens["input_ids"] +
                    question_tokens["input_ids"][:max_question_len] +
                    self.prompt3_tokens["input_ids"] +
                    label_tokens["input_ids"] +
                    [self.end_token_ids]
            )
#             print("input_ids:", input_ids)
            
            if idx == 0:
                print("example:")
                print(self.tokenizer.decode(input_ids))
            attention_mask = (
                    self.prompt1_tokens["attention_mask"] +
                    context_tokens["attention_mask"][:max_context_len] +
                    self.prompt2_tokens["attention_mask"] +
                    question_tokens["attention_mask"][:max_question_len] +
                    self.prompt3_tokens["attention_mask"] +
                    label_tokens["attention_mask"] +
                    [1]
            )

            labels = (
                    [-100] * (len(input_ids) - len(label_tokens["input_ids"]) - 1)
                    + label_tokens["input_ids"]
                    + [self.end_token_ids]
            )
            
            self.input_data.append({"input_ids": input_ids,
                                    "attention_mask": attention_mask,
                                    "labels": labels, })
        print("tokenization done!")

    def __getitem__(self, idx):
        # 返回一个字典，包含 input_ids, attention_mask, 以及 labels (如果有的话)
        return self.input_data[idx]

    def __len__(self):
        return len(self.input_data)

In [13]:
class CustomTrainer(Trainer):
    def training_step(self, model, inputs):
        model.train()
        inputs = self._prepare_inputs(inputs)

        # 前向传播，获取模型输出并计算损失
        outputs = model(**inputs)
        loss = outputs.loss

        # 手动调用反向传播
        loss.backward()

        # 打印每一层的参数和值
        for name, param in model.named_parameters():
            if param.grad is not None:
                print(f"Layer: {name}, Parameter value mean: {param.data.mean().item()}, Gradient mean: {param.grad.abs().mean().item()}")
            else:
                print(f"Layer: {name}, Parameter value mean: {param.data.mean().item()}, Gradient: None")

        # 梯度裁剪（如果需要）
        if self.args.max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), self.args.max_grad_norm)

        return loss.detach()

In [ ]:
'''
main
'''

from peft import LoraConfig, TaskType, get_peft_model
from swanlab.integration.huggingface import SwanLabCallback
import swanlab
from transformers import TrainerCallback
import torch
import gc

model=None
dataset_name = "hotpot"
post="train"
da = 0
output_dir = os.path.join("./output/Qwen_final", dataset_name+str(da))
split_ratio = 1.0
    
model, tokenizer = load_model()
train_dataset = CustomDataset(dataset_name, tokenizer, post, da)
eval_dataset = copy.deepcopy(train_dataset)
train_dataset.input_data = train_dataset.input_data[:int(split_ratio*len(train_dataset))]
eval_dataset.input_data = eval_dataset.input_data [int(split_ratio*len(eval_dataset)):]
print("train_dataset len:", len(train_dataset))
print("eval_dataset len:", len(eval_dataset))


class CustomTrainer(Trainer):
    def training_step(self, model, inputs, num_items_in_batch):
#         model.zero_grad()
        model.train()
#         if torch.cuda.is_available():
#             print(f"111Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
#             print(f"111Cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

        
        inputs = self._prepare_inputs(inputs)
#         if torch.cuda.is_available():
#             print(f"222Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
#             print(f"222Cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

        
        # 前向传播，获取模型输出并计算损失
        outputs = model(**inputs)
#         if torch.cuda.is_available():
#             print(f"333Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
#             print(f"333Cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

        
        loss = outputs.loss
#         if torch.cuda.is_available():
#             print(f"444Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
#             print(f"444Cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

        

        # 手动调用反向传播
        loss.backward()

        # 打印每一层的参数和值
        if num_items_in_batch % 50 == 0:
            for name, param in model.named_parameters():
                if "base_model.model.model.layers.0.self_attn.v_proj.lora_B" in name and param.grad is not None:
                    print(f"Layer: {name}, Parameter value mean: {param.data.mean().item()}, Gradient mean: {param.grad.abs().mean().item()}")

        # 梯度裁剪（如果需要）
#         if self.args.max_grad_norm is not None:
#             torch.nn.utils.clip_grad_norm_(model.parameters(), self.args.max_grad_norm)
        
#         gc.collect()
#         torch.cuda.empty_cache()
        
        return loss.detach()



def lora_config():
# set lora
    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        inference_mode=False,  # 训练模式
        r=8,  # Lora 秩
        lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
        lora_dropout=0.1,  # Dropout 比例
    )
    return config

def load_swanlab_callback():
    swanlab_callback = SwanLabCallback(
        project="Qwen2-finetune_hotpot0",
        experiment_name="Qwen2-0.5B_final"+dataset_name+post+str(da),
        description="Qwen2-finetune",
    )
    return swanlab_callback


class LoRAGradientPrinterCallback(TrainerCallback):
    def on_step_end(self, args, state, control, model=None, **kwargs):
        if torch.cuda.is_available():
            print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            print(f"Cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

        
        for name, param in model.named_parameters():
            if "base_model.model.model.layers.0.self_attn.v_proj.lora_B" in name and param.grad is not None:
                print(f"Layer: {name}, Parameter value mean: {param.data.mean().item()}, Gradient mean: {param.grad.abs().mean().item()}")

                
args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    logging_steps=10,
    num_train_epochs=3,
    save_steps=100,
#     eval_steps=100,
#     max_grad_norm=1.0,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
#     optim="adamw_torch",
#     fp16=True,  # 禁用自动混合精度
    bf16=True,
)


config = lora_config()
model = get_peft_model(model, config)
swanlab_callback = load_swanlab_callback()

# print(model)

print("******************before training******************")
for name, param in model.named_parameters():
    if "base_model.model.model.layers.0.self_attn.v_proj.lora_B" in name:
    #             if "lora" in name:  # LoRA 层的参数通常带有 "lora" 标记, -> has no gradient
    #             if param.requires_grad and param.grad is None:
        print(f" Layer: {name},  {param.data.mean()}")
    
    
trainer = CustomTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
#     callbacks = [LoRAGradientPrinterCallback()],
    callbacks=[swanlab_callback],
)
print(model)
trainer.train()
trainer.model.save_pretrained(output_dir)
# outputs = model(torch.tensor([[1,2,3,4]]).cuda())
# print(outputs) 

using device: cuda
loading from:./dataset/hotpotQA/hotpot_train_v1.1.json.train_0


5000it [00:00, 11170.96it/s]


train_num:5000
begin tokenize


  0%|▌                                                                                                                                            | 19/5000 [00:00<00:27, 184.11it/s]

example:
You are Qwen, created by Alibaba Cloud. You are a helpful assistant. Now please answer the question according to the given context. Just give me answer without explanation. Context and question are as follows: 
context:David Gregory Phillips is a fictional character on the CBS crime drama "", portrayed by David Berman, who also serves as head researcher for the series. From Season 10 onwards David Berman has been credited in the opening titles.Lookout Mountain, Lookout Sea is the sixth and final studio album by American indie rock band Silver Jews, released on 17 June 2008 on Drag City. It was recorded at Marble Valley of Lexington, Virginia and Lake Fever Productions of Nashville, Tennessee. Silver Jews records are known for featuring different casts of musicians. This album features the touring band of lead singer, David Berman, including his wife Cassie. Berman has said that the album is "really different" compared to previous ones.Benjamin "Bugsy" Siegel (February 28, 1906

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5000/5000 [00:14<00:00, 338.68it/s]


tokenization done!
train_dataset len: 5000
eval_dataset len: 0
******************before training******************
 Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight,  0.0
PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
              

Step,Training Loss
10,11.945100
20,10.187500
30,9.318600
40,8.183800
50,7.992100
60,8.672600
70,8.426000
80,7.296700
90,7.386600
100,6.227800


Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Parameter value mean: 3.9021315387799405e-06, Gradient mean: 0.035327136516571045
Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Parameter value mean: 3.9021315387799405e-06, Gradient mean: 0.07013736665248871
Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Parameter value mean: 3.9021315387799405e-06, Gradient mean: 0.12157644331455231
Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Parameter value mean: 3.9021315387799405e-06, Gradient mean: 0.14196664094924927
Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Parameter value mean: 3.9021315387799405e-06, Gradient mean: 0.15510861575603485
Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Parameter value mean: 3.9021315387799405e-06, Gradient mean: 0.15678608417510986
Layer: base_model.model.model.layers.0.

Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Parameter value mean: 4.503875061345752e-06, Gradient mean: 0.20512376725673676
Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Parameter value mean: 4.503875061345752e-06, Gradient mean: 0.2048962116241455
Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Parameter value mean: 4.503875061345752e-06, Gradient mean: 0.21331626176834106
Layer: base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Parameter value mean: 4.503875061345752e-06, Gradient mean: 0.21423736214637756
swanlab: Step 50 on key train/loss already exists, ignored.
swanlab: Step 50 on key train/grad_norm already exists, ignored.
swanlab: Step 50 on key train/learning_rate already exists, ignored.
swanlab: Step 50 on key train/epoch already exists, ignored.
swanlab: Step 60 on key train/loss already exists, ignored.
swanlab: Step 60 on key train/grad_norm already ex